# IVP Group 33 — Hindi MNIST Training
Run cells top to bottom. Make sure your Google Drive has this structure:
```
MyDrive/IVP-Group33/
  yolo26m-cls.pt
  data/
    train.csv
    test.csv
    train/train/<0-9 folders>
    test/test/<images>
```

In [ ]:
# Check we have a GPU
!nvidia-smi

In [ ]:
!pip install ultralytics -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import pandas as pd
from ultralytics import YOLO

ROOT      = Path('/content/drive/MyDrive/IVP-Group33')
DATA      = ROOT / 'data'
TRAIN_SRC = DATA / 'train' / 'train'
TEST_SRC  = DATA / 'test'  / 'test'
WEIGHTS   = ROOT / 'yolo26m-cls.pt'

# Sanity check paths exist
for p in [WEIGHTS, TRAIN_SRC, TEST_SRC, DATA / 'test.csv']:
    print(p, '✓' if p.exists() else '✗ NOT FOUND')

In [ ]:
model = YOLO(str(WEIGHTS))

model.train(
    data=str(TRAIN_SRC),
    epochs=50,
    degrees=10,   # small rotations — realistic handwriting tilt
    fliplr=0.0,   # no horizontal flips — flipping digits changes their identity
    hsv_s=0.0,    # no saturation aug — data is grayscale
    hsv_h=0.0,    # no hue aug
    mosaic=0.0,   # no mosaic — wrong augmentation type for single-digit task
    device=0,     # use GPU
)

In [ ]:
# Use best checkpoint for prediction
best = Path('runs/classify/train/weights/best.pt')
model = YOLO(str(best))

results = model.predict(source=str(TEST_SRC), save=False)
pred_map = {Path(r.path).stem: int(r.probs.top1) for r in results}

test_csv = pd.read_csv(DATA / 'test.csv')
test_csv['Category'] = test_csv['Id'].astype(str).map(pred_map)

out = ROOT / 'submission.csv'
test_csv.to_csv(out, index=False)
print(f'Saved {out} ({len(test_csv)} rows)')

In [ ]:
# Optional: download submission.csv directly to your computer
from google.colab import files
files.download(str(ROOT / 'submission.csv'))